# Acinar Analysis
*Unified pipeline for all 3D acinar image quantification*

This notebook provides an easy interface to run all acinar analyses from `acinar_analysis.py`.

## Available analyses
| Analysis | What it measures | Required inputs |
|---|---|---|
| `acinus_shape` | Acinus volume (µm³) and roundness | Image only |
| `cell_nuclear_shape` | Cell/nuclear volume, roundness, neighbours | Image + nuclear mask + membrane mask |
| `protein_polarisation` | Protein intensity vs radial distance from centre | Image + protein channel |
| `apoptosis` | C3+ cell count, spatial distribution, nuclei count | Image + C3 mask + DAPI mask |
| `protein_proximity` | Protein intensity near dying vs non-dying cells | Image + C3/DAPI masks + proximity channel |

## How to use
1. **Run cell 1** to import the module
2. **Fill in your paths** in the Setup cell
3. **Pick your analysis** — run the corresponding section
4. Results are returned as DataFrames and optionally saved to CSV

In [1]:
# Cell 1: Import the module
import acinar_analysis as aa
import pandas as pd
import pathlib

print("Available analyses:", aa.VALID_ANALYSES)

ModuleNotFoundError: No module named 'joblib'

---
## Setup: Define your paths and channels

Edit the variables below to match your data. You only need to fill in the paths/channels relevant to the analysis you want to run.

**Channel indices** correspond to the order of channels in your multi-channel TIFF (0-indexed).

In [ ]:
# ============================================================
# EDIT THESE TO MATCH YOUR DATA
# ============================================================

# --- Image paths ---
# For single-image analysis:
image_path = r"path\to\your\image.tif"

# For batch analysis (folder of images):
image_dir = r"path\to\your\image_folder"
file_extension = "tif"  # e.g. "tif", "tiff"

# --- Channel indices (0-indexed) ---
dapi_channel = 0        # Nuclear channel (DAPI) — almost always needed
membrane_channel = 2    # Membrane/cytoplasmic marker (e.g. CAAX) — set to None if not available
protein_channel = 1     # Protein of interest (for polarisation analysis) — set to None if not needed
c3_channel = 3          # Cleaved caspase-3 channel — set to None if not needed
proximity_protein_channel = None  # Protein for proximity analysis (e.g. integrin) — set to None if not needed

# --- Segmentation mask paths (for analyses that require them) ---
# Single image masks:
nuclear_mask_path = None    # e.g. r"path\to\nuclear_segmentation.tif"
membrane_mask_path = None   # e.g. r"path\to\membrane_segmentation.tif"
c3_mask_path = None         # e.g. r"path\to\c3_binary.tif"
dapi_mask_path = None       # e.g. r"path\to\dapi_binary.tif"

# Batch mask directories (one mask per image, matched alphabetically):
nuclear_mask_dir = None     # e.g. r"path\to\nuclear_masks"
membrane_mask_dir = None    # e.g. r"path\to\membrane_masks"
c3_mask_dir = None          # e.g. r"path\to\c3_masks"
dapi_mask_dir = None        # e.g. r"path\to\dapi_masks"

# --- Output ---
output_csv = "acinar_results.csv"  # Output filename for batch analysis
n_jobs = 3  # Number of parallel workers for batch processing

print("Setup complete.")

---
## Option A: Acinus Shape (volume & roundness)
**Requires:** Image only (DAPI + membrane channels)

No segmentation masks needed.

In [ ]:
# --- Single image ---
results = aa.analyse_image(
    image_path,
    analyses=["acinus_shape"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
)
results["acinus_shape"]

In [ ]:
# --- Batch (whole folder) ---
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["acinus_shape"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    output_csv=output_csv,
    n_jobs=n_jobs,
)
batch_results["acinus_shape"].head()

---
## Option B: Cell & Nuclear Shape
**Requires:** Image + binary nuclear segmentation mask + binary membrane segmentation mask

You **must** set `nuclear_mask_path` and `membrane_mask_path` (or the `_dir` equivalents for batch) in the Setup cell.

In [ ]:
# --- Single image ---
results = aa.analyse_image(
    image_path,
    analyses=["cell_nuclear_shape"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    nuclear_mask_path=nuclear_mask_path,
    membrane_mask_path=membrane_mask_path,
)
results["cell_nuclear_shape"]

In [ ]:
# --- Batch (whole folder) ---
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["cell_nuclear_shape"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    nuclear_mask_dir=nuclear_mask_dir,
    membrane_mask_dir=membrane_mask_dir,
    output_csv=output_csv,
    n_jobs=n_jobs,
)
batch_results["cell_nuclear_shape"].head()

---
## Option C: Protein Polarisation
**Requires:** Image + protein channel index

Set `protein_channel` in the Setup cell.

In [ ]:
# --- Single image ---
results = aa.analyse_image(
    image_path,
    analyses=["protein_polarisation"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    protein_channel=protein_channel,
)
df_polar = results["protein_polarisation"]
df_polar.head()

In [ ]:
# Quick plot: protein intensity vs normalised distance from centre
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df_polar["rounded_distance"], df_polar["protein_intensity"] / df_polar["protein_intensity"].max())
ax.set_xlabel("Normalised distance from centre (0 = edge, 1 = centre)")
ax.set_ylabel("Normalised protein intensity")
ax.set_title("Protein Polarisation")
plt.tight_layout()
plt.show()

In [ ]:
# --- Batch (whole folder) ---
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["protein_polarisation"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    protein_channel=protein_channel,
    output_csv=output_csv,
    n_jobs=n_jobs,
)
batch_results["protein_polarisation"].head()

---
## Option D: Apoptosis Quantification (C3 counting)
**Requires:** Image + binary C3 mask + binary DAPI mask

Set `c3_mask_path`, `dapi_mask_path`, and `c3_channel` in the Setup cell.

In [ ]:
# --- Single image ---
results = aa.analyse_image(
    image_path,
    analyses=["apoptosis"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    c3_channel=c3_channel,
    c3_mask_path=c3_mask_path,
    dapi_mask_path=dapi_mask_path,
)
results["apoptosis"]

In [ ]:
# --- Batch (whole folder) ---
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["apoptosis"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    c3_channel=c3_channel,
    c3_mask_dir=c3_mask_dir,
    dapi_mask_dir=dapi_mask_dir,
    output_csv=output_csv,
    n_jobs=n_jobs,
)
batch_results["apoptosis"].head()

---
## Option E: Protein Proximity (intensity near dying vs non-dying cells)
**Requires:** Image + binary C3 mask + binary DAPI mask + proximity protein channel

Set `c3_mask_path`, `dapi_mask_path`, `c3_channel`, and `proximity_protein_channel` in the Setup cell.

In [ ]:
# --- Single image ---
# You can adjust search_radius_um to control the neighbourhood size around each cell
results = aa.analyse_image(
    image_path,
    analyses=["protein_proximity"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    c3_channel=c3_channel,
    proximity_protein_channel=proximity_protein_channel,
    c3_mask_path=c3_mask_path,
    dapi_mask_path=dapi_mask_path,
    search_radius_um=5.0,  # adjust as needed
)
results["protein_proximity"]

In [ ]:
# --- Batch (whole folder) ---
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["protein_proximity"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    c3_channel=c3_channel,
    proximity_protein_channel=proximity_protein_channel,
    c3_mask_dir=c3_mask_dir,
    dapi_mask_dir=dapi_mask_dir,
    search_radius_um=5.0,
    output_csv=output_csv,
    n_jobs=n_jobs,
)
batch_results["protein_proximity"].head()

---
## Option F: Run multiple analyses at once
You can combine any analyses in a single call — results are returned as a dict keyed by analysis name.

In [ ]:
# Example: acinus shape + protein polarisation together
results = aa.analyse_image(
    image_path,
    analyses=["acinus_shape", "protein_polarisation"],
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    protein_channel=protein_channel,
)

for name, df in results.items():
    print(f"\n=== {name} ===")
    display(df.head())

In [ ]:
# Batch: multiple analyses at once
# Each analysis gets its own output CSV: e.g. results_acinus_shape.csv, results_protein_polarisation.csv
batch_results = aa.batch_analyse(
    image_dir=image_dir,
    analyses=["acinus_shape", "protein_polarisation"],
    file_extension=file_extension,
    dapi_channel=dapi_channel,
    membrane_channel=membrane_channel,
    protein_channel=protein_channel,
    output_csv=output_csv,
    n_jobs=n_jobs,
)

for name, df in batch_results.items():
    print(f"\n=== {name}: {len(df)} rows ===")
    display(df.head())

---
## Troubleshooting

| Error | What it means | Fix |
|---|---|---|
| `ValueError: ... requires 'nuclear_mask_path' and 'membrane_mask_path'` | You're running `cell_nuclear_shape` without masks | Set the mask paths in the Setup cell |
| `ValueError: ... requires 'c3_mask_path' and 'dapi_mask_path'` | You're running `apoptosis` or `protein_proximity` without masks | Set the mask paths in the Setup cell |
| `ValueError: ... requires 'protein_channel'` | You're running `protein_polarisation` without specifying the channel | Set `protein_channel` in the Setup cell |
| `ValueError: Unknown analyses` | Typo in analysis name | Use one of: `acinus_shape`, `cell_nuclear_shape`, `protein_polarisation`, `apoptosis`, `protein_proximity` |
| `ValueError: Mask count mismatch` | Different number of images and masks in batch mode | Check that mask folders have the same number of TIFFs as the image folder |
| `FileNotFoundError: No .tif files found` | Wrong directory or file extension | Check `image_dir` and `file_extension` |